# Sensibilidad de la doble barrera de acceso al hospital

¿Qué municipios siguen concentrando casos confirmados al usar umbrales de **90, 120 y 150 minutos**? La doble barrera combina un traslado al hospital superior al umbral con una respuesta explícita de ausencia de transporte foráneo en RESLOC.

La población evaluada tiene un tiempo hospitalario y una respuesta de sí o no sobre transporte. La cobertura se calcula sobre toda la población municipal del MVP. Se mantiene fija entre escenarios porque cambiar el umbral no cambia qué datos están disponibles. Una cobertura baja limita la comparación: los conteos describen los casos confirmados, sin extrapolar a la población sin información.

In [1]:
from pathlib import Path
import sys

import pandas as pd

inicio = Path.cwd().resolve()
raiz = next(p for p in (inicio, *inicio.parents) if (p / "datos_mvp/manifest.json").is_file())
sys.path.insert(0, str(raiz / "src"))

from health_data import territorio

salud = raiz / "datos_mvp/salud"
accesibilidad = pd.read_parquet(salud / "accesibilidad_localidades.parquet")
transporte = pd.read_parquet(salud / "inegi-resloc-001_localidades.parquet")
marginacion = pd.read_parquet(salud / "conapo-marg-001_localidades.parquet")
umbrales = (90, 120, 150)
escenarios = []
for corte in umbrales:
    localidades = territorio.preparar_localidades(
        accesibilidad, marginacion, transporte, umbral_centro=60, umbral_hospital=corte,
    )
    municipios = territorio.resumir_municipios(localidades)
    escenarios.append(municipios.assign(umbral_min=corte))

panel = pd.concat(escenarios, ignore_index=True)
totales = panel.groupby("umbral_min")[
    ["doble_barrera", "poblacion_doble_evaluable", "population"]
].sum(min_count=1)
totales["cobertura_pct"] = 100 * totales["poblacion_doble_evaluable"] / totales["population"]
display(totales.round(1))

,doble_barrera,poblacion_doble_evaluable,population,cobertura_pct
umbral_min,,,,
90,13229.0,297784,2944824,10.1
120,8237.0,297784,2944824,10.1
150,5897.0,297784,2944824,10.1


## Comparación municipal

Los tres conteos se muestran por separado. **0** indica que hubo población evaluada y ningún caso confirmado. **Sin dato** indica que ninguna persona pudo evaluarse. Se señala cobertura baja por debajo de **50 %** como regla de lectura de este análisis. El corte es orientativo.

Para identificar resultados que se mantienen, se compara el grupo de los diez municipios con más casos confirmados en cada escenario. Se incluyen empates en el décimo lugar y se excluyen ceros y valores ausentes de la clasificación. Un municipio puede seguir entre los diez primeros aunque su conteo cambie.

In [2]:
conteos = panel.pivot(index="cvegeo", columns="umbral_min", values="doble_barrera")
posiciones = conteos.where(conteos.gt(0)).rank(method="min", ascending=False)
num_escenarios_top = posiciones.le(10).sum(axis=1)
base = escenarios[0].set_index("cvegeo")

comparacion = base[["municipio", "population", "poblacion_doble_evaluable", "pct_doble_evaluable"]].join(
    conteos.rename(columns={corte: f"Casos >{corte} min" for corte in umbrales}).astype("Int64")
)
comparacion["Escenarios en el top 10"] = num_escenarios_top
comparacion["Cobertura"] = "50 % o más"
comparacion.loc[base["pct_doble_evaluable"].lt(50), "Cobertura"] = "Baja (<50 %)"
comparacion.loc[base["poblacion_doble_evaluable"].eq(0), "Cobertura"] = "Sin datos evaluables"
comparacion["Sensibilidad del conteo"] = "Se mantiene"
comparacion.loc[conteos.nunique(axis=1).gt(1), "Sensibilidad del conteo"] = "Cambia"
comparacion.loc[conteos.isna().all(axis=1), "Sensibilidad del conteo"] = "Sin dato"

comparacion = comparacion.rename(columns={
    "population": "Población total",
    "poblacion_doble_evaluable": "Población evaluada",
    "pct_doble_evaluable": "Cobertura %",
}).sort_values(["Escenarios en el top 10", "Casos >120 min", "municipio"], ascending=[False, False, True])
columnas_casos = [f"Casos >{corte} min" for corte in umbrales]
comparacion_visible = comparacion.round({"Cobertura %": 1}).astype({c: "object" for c in columnas_casos})
comparacion_visible[columnas_casos] = comparacion_visible[columnas_casos].fillna("Sin dato")
with pd.option_context("display.max_rows", None):
    display(comparacion_visible)

,municipio,Población total,Población evaluada,Cobertura %,Casos >90 min,Casos >120 min,Casos >150 min,Escenarios en el top 10,Cobertura,Sensibilidad del conteo
cvegeo,,,,,,,,,,
26060,Sáric,2058,1319,64.1,1309,1304,1013,3,50 % o más,Cambia
26069,Yécora,4793,2204,46.0,988,988,835,3,Baja (<50 %),Cambia
26052,Sahuaripa,5257,1445,27.5,921,712,492,3,Baja (<50 %),Cambia
26061,Soyopa,1368,1175,85.9,804,599,592,3,50 % o más,Cambia
26047,Pitiquito,9122,1117,12.2,563,560,559,3,Baja (<50 %),Cambia
26040,Nácori Chico,1531,711,46.4,711,521,521,3,Baja (<50 %),Cambia
26051,Rosario,4830,1839,38.1,1047,317,262,3,Baja (<50 %),Cambia
26003,Alamos,24976,13809,55.3,1792,659,52,2,50 % o más,Cambia
26005,Arivechi,1177,522,44.4,410,374,374,2,Baja (<50 %),Cambia


In [3]:
persistentes = comparacion.loc[comparacion["Escenarios en el top 10"].eq(len(umbrales))]
dependen_del_umbral = comparacion.loc[comparacion["Escenarios en el top 10"].between(1, len(umbrales) - 1)]
print("Entre los diez con más casos en los tres escenarios:", ", ".join(persistentes["municipio"]) or "ninguno")
print("Entre los diez con más casos solo en algunos escenarios:", ", ".join(dependen_del_umbral["municipio"]) or "ninguno")
print("Municipios con conteos que cambian:", comparacion["Sensibilidad del conteo"].eq("Cambia").sum())
print("Municipios con cobertura baja:", comparacion["Cobertura"].eq("Baja (<50 %)").sum())
print("Municipios sin datos evaluables:", comparacion["Cobertura"].eq("Sin datos evaluables").sum())
display(persistentes[["municipio", "Cobertura %", "Cobertura", *columnas_casos]].round({"Cobertura %": 1}))

Entre los diez con más casos en los tres escenarios: Sáric, Yécora, Sahuaripa, Soyopa, Pitiquito, Nácori Chico, Rosario
Entre los diez con más casos solo en algunos escenarios: Alamos, Arivechi, Cajeme, Caborca, Arizpe, Quiriego
Municipios con conteos que cambian: 47
Municipios con cobertura baja: 60
Municipios sin datos evaluables: 3


,municipio,Cobertura %,Cobertura,Casos >90 min,Casos >120 min,Casos >150 min
cvegeo,,,,,,
26060,Sáric,64.1,50 % o más,1309,1304,1013
26069,Yécora,46.0,Baja (<50 %),988,988,835
26052,Sahuaripa,27.5,Baja (<50 %),921,712,492
26061,Soyopa,85.9,50 % o más,804,599,592
26047,Pitiquito,12.2,Baja (<50 %),563,560,559
26040,Nácori Chico,46.4,Baja (<50 %),711,521,521
26051,Rosario,38.1,Baja (<50 %),1047,317,262


## Lectura del corte 0.1.1

Con 90, 120 y 150 minutos se confirman 13,229, 8,237 y 5,897 personas con doble barrera, respectivamente. Se pudieron evaluar 297,784 personas, el 10.1 % de la población. Tres municipios quedaron sin población evaluable.

Sáric, Yécora, Sahuaripa, Soyopa, Pitiquito, Nácori Chico y Rosario aparecen entre los diez primeros en los tres escenarios. Entre ellos, solo Sáric y Soyopa tienen al menos 50 % de cobertura. Otros seis municipios aparecen entre los diez primeros únicamente con algunos umbrales, como muestra la lista anterior.

La estabilidad se refiere a los casos confirmados con los datos disponibles. Los conteos no estiman la doble barrera entre las personas sin información.